In [1]:
!pip install -q duckdb

In [9]:
import duckdb
import pandas as pd

# 1. Pehle line 5901-5910 inspect karein (agar kuch ajeeb dikhe toh samajh aayega)
with open("/content/olist_orders_dataset.csv", "r", encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

print("Total lines:", len(lines))
for i in range(5900, min(5910, len(lines))):
    print(f"Line {i+1}: {lines[i].strip()}")

# 2. Safe load: bad/malformed rows ko handle karte hue CSV padhna
orders = pd.read_csv(
    "/content/olist_orders_dataset.csv",
    engine="python",
    on_bad_lines="warn"  # agar warning bahut zyada aaye toh "skip" kar dein
)
items = pd.read_csv("/content/olist_order_items_dataset.csv")

# 3. DuckDB mein register aur SQL query run karna
con = duckdb.connect()
con.register("orders", orders)
con.register("items", items)

df_result = con.execute("""
    SELECT
        DATE_TRUNC('month', CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS order_month,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(SUM(i.price), 2) AS gross_revenue
    FROM orders o
    JOIN items i ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
    ORDER BY 1;
""").df()

df_result.head(10)

Total lines: 169358
Line 5901: dcfdf4c257739648a81742a5ea7f8a07,"264cfae73963c9e7444cd4ecb6bf63fe",delivered,2018-04-19 23:24:21,2018-04-24 18:22:33,2018-04-24 11:06:37,2018-04-28 11:36:49,2018-06-08 00:00:00
Line 5902: "3641f9fa33b23a96ddef8d28f440d8db",cdf31148c659090167c2868425d2a72f,delivered,2018-03-27 08:10:06,2018-03-27 08:25:56,2018-03-27 21:59:56,2018-04-19 21:42:49,2018-04-23 00:00:00
Line 5903: "8d7e588f9edf5e801cf455bf1961cffb","4f828355a45997653584184482dd1c2a",delivered,2018-06-19 14:01:34,2018-06-19 14:17:44,2018-06-21 13:01:00,2018-06-29 21:08:28,2018-07-13 00:00:00
Line 5904: "1e7bb5d072b8543a1c42507c4b5583eb",daac60ed0587e8455cbfa9799d658f27,delivered,2018-01-26 16:47:34,2018-01-26 17:22:24,2018-01-29 23:29:12,2018-01-31 19:59:25,2018-02-09 00:00:00
Line 5905: dde3f065ba9416ef2eb1e9987a28ac59,"944539610a067aa2b23935a72f94c768",delivered,2018-03-26 18:21:01,2018-03-27 03:55:45,2018-04-04 21:44:50,2018-04-09 20:56:37,2018-04-12 00:00:00f31cc598dd8b9","5519a510e5eb7dff3e

/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 29529: ',' expected after '"'

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 53150: ',' expected after '"'

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 106310: ',' expected after '"'

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 5905: Expected 8 fields in line 5905, saw 15

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 17717: Expected 8 fields in line 17717, saw 10

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 35432: Expected 8 fields in line 35432, saw 12

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 47243: Expected 8 fields in line 47243, saw 12

  orders = pd.read_csv(
/tmp/ipykernel_670/462244913.py:13: ParserWarning: Skipping line 64963: Expected 8 fields in 

,order_month,total_orders,gross_revenue
0,2016-09-01,1,269.94
1,2016-10-01,265,66920.28
2,2016-12-01,1,21.80
3,2017-01-01,750,183593.25
4,2017-02-01,1653,392652.19
5,2017-03-01,2545,617546.20
6,2017-04-01,2303,584586.29
7,2017-05-01,3546,817891.94
8,2017-06-01,3135,712639.09
9,2017-07-01,3872,824081.95


In [10]:
products = pd.read_csv("/content/olist_products_dataset.csv", engine="python", on_bad_lines="skip")
customers = pd.read_csv("/content/olist_customers_dataset.csv", engine="python", on_bad_lines="skip")
payments = pd.read_csv("/content/olist_order_payments_dataset.csv", engine="python", on_bad_lines="skip")
cat_trans = pd.read_csv("/content/product_category_name_translation.csv", engine="python", on_bad_lines="skip")

con.register("products", products)
con.register("customers", customers)
con.register("payments", payments)
con.register("cat_trans", cat_trans)

print("All tables registered successfully!")

All tables registered successfully!


# Query-1:Monthly Revenue & Order Volume Trend

In [11]:
"""How have monthly delivered orders and gross merchandise revenue evolved over time across the marketplace?"""
q1_monthly_trend = con.execute("""
    SELECT
        DATE_TRUNC('month', CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS order_month,
        COUNT(DISTINCT o.order_id) AS total_delivered_orders,
        COUNT(i.order_item_id) AS total_items_sold,
        ROUND(SUM(i.price), 2) AS gross_product_revenue,
        ROUND(SUM(i.price) / NULLIF(COUNT(DISTINCT o.order_id), 0), 2) AS avg_order_value
    FROM orders o
    JOIN items i
        ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1
    ORDER BY 1;
""").df()

q1_monthly_trend

,order_month,total_delivered_orders,total_items_sold,gross_product_revenue,avg_order_value
0,2016-09-01,1,6,269.94,269.94
1,2016-10-01,265,530,66920.28,252.53
2,2016-12-01,1,2,21.80,21.80
3,2017-01-01,750,1545,183593.25,244.79
4,2017-02-01,1653,3154,392652.19,237.54
5,2017-03-01,2545,4919,617546.20,242.65
6,2017-04-01,2303,4407,584586.29,253.84
7,2017-05-01,3546,6812,817891.94,230.65
8,2017-06-01,3135,5941,712639.09,227.32
9,2017-07-01,3872,7558,824081.95,212.83


Business Insight:

Filtering strictly on order_status = 'delivered' prevents inflated revenue from cancelled or undelivered orders, giving a true baseline of realized sales.

Tracking total delivered orders alongside Average Order Value (AOV) reveals whether monthly revenue growth is driven by new customer/order volume or higher basket sizes per order.

# Query-2:Month-over-Month (MoM) Revenue & Order Growth

In [12]:
"""What is the month-over-month (MoM) percentage growth in delivered gross revenue and order
volume, and which months experienced the sharpest acceleration or slowdown?"""
q2_mom_growth = con.execute("""
    WITH monthly_metrics AS (
        SELECT
            DATE_TRUNC('month', CAST(o.order_purchase_timestamp AS TIMESTAMP)) AS order_month,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(i.price), 2) AS gross_revenue
        FROM orders o
        JOIN items i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        order_month,
        total_orders,
        gross_revenue,
        LAG(gross_revenue) OVER (ORDER BY order_month) AS prev_month_revenue,
        ROUND(
            ((gross_revenue - LAG(gross_revenue) OVER (ORDER BY order_month)) /
             NULLIF(LAG(gross_revenue) OVER (ORDER BY order_month), 0)) * 100,
            2
        ) AS mom_revenue_growth_pct,
        ROUND(
            ((total_orders - LAG(total_orders) OVER (ORDER BY order_month)) * 100.0) /
             NULLIF(LAG(total_orders) OVER (ORDER BY order_month), 0),
            2
        ) AS mom_order_growth_pct
    FROM monthly_metrics
    ORDER BY order_month;
""").df()

q2_mom_growth

,order_month,total_orders,gross_revenue,prev_month_revenue,mom_revenue_growth_pct,mom_order_growth_pct
0,2016-09-01,1,269.94,NaN,NaN,NaN
1,2016-10-01,265,66920.28,269.94,24690.80,26400.00
2,2016-12-01,1,21.80,66920.28,-99.97,-99.62
3,2017-01-01,750,183593.25,21.80,842070.87,74900.00
4,2017-02-01,1653,392652.19,183593.25,113.87,120.40
5,2017-03-01,2545,617546.20,392652.19,57.28,53.96
6,2017-04-01,2303,584586.29,617546.20,-5.34,-9.51
7,2017-05-01,3546,817891.94,584586.29,39.91,53.97
8,2017-06-01,3135,712639.09,817891.94,-12.87,-11.59
9,2017-07-01,3872,824081.95,712639.09,15.64,23.51


Business Insight:

Using LAG() over a monthly CTE isolates true period-over-period momentum without manual self-joins, making peak promotional/holiday surges and post-peak dips immediately visible.

Comparing mom_revenue_growth_pct against mom_order_growth_pct shows whether revenue spikes came from higher order frequency or higher-priced items.

# Query 3: Top Revenue-Generating Product Categories

In [17]:
"""Which product categories generate the highest realized revenue and order volume, and
how do their average item prices and revenue share compare across the catalog?"""
q3_top_categories = con.execute("""
    WITH category_sales AS (
        SELECT
            COALESCE(t.product_category_name_english, p.product_category_name, 'Unknown') AS category,
            COUNT(DISTINCT o.order_id) AS total_orders,
            COUNT(i.order_item_id) AS total_units_sold,
            ROUND(SUM(i.price), 2) AS total_revenue,
            ROUND(AVG(i.price), 2) AS avg_item_price
        FROM orders o
        JOIN items i
            ON o.order_id = i.order_id
        JOIN products p
            ON i.product_id = p.product_id
        LEFT JOIN cat_trans t
            ON p.product_category_name = t.product_category_name
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        category,
        total_orders,
        total_units_sold,
        total_revenue,
        avg_item_price,
        ROUND(
            (total_revenue * 100.0) / NULLIF(SUM(total_revenue) OVER (), 2),
            2
        ) AS revenue_share_pct
    FROM category_sales
    ORDER BY total_revenue DESC
    LIMIT 10;
""").df()

q3_top_categories

,category,total_orders,total_units_sold,total_revenue,avg_item_price,revenue_share_pct
0,health_beauty,8647,16139,2097786.70,129.98,9.35
1,watches_gifts,5495,9959,1981032.79,198.92,8.82
2,bed_bath_table,9269,18687,1735934.25,92.90,7.73
3,sports_leisure,7530,14335,1622836.60,113.21,7.23
4,computers_accessories,6529,13015,1512622.28,116.22,6.74
5,furniture_decor,6307,13924,1212425.23,87.07,5.40
6,housewares,5742,11583,1044279.11,90.16,4.65
7,cool_stuff,3558,6260,1022921.30,163.41,4.56
8,auto,3810,6942,965685.14,139.11,4.30
9,garden_tools,3447,7280,801995.02,110.16,3.57


Business Insight:

Joining cat_trans with COALESCE standardizes Portuguese category labels into readable English names while preventing any records with missing translations from dropping out.

Contrasting revenue_share_pct with avg_item_price highlights whether a category leads through high-ticket items or high-volume repeat demand—helping prioritize merchandising, stock availability, and margin focus.

# Query 4: Repeat Customer Rate & Customer Spend Comparison

In [15]:
"""What share of unique buyers place repeat orders versus one-time orders, and
 how does the average total spend of repeat customers compare with one-time customers?"""

q4_repeat_customers = con.execute("""
    WITH customer_summary AS (
        SELECT
            c.customer_unique_id,
            COUNT(DISTINCT o.order_id) AS delivered_orders,
            ROUND(SUM(i.price), 2) AS total_spent
        FROM orders o
        JOIN customers c
            ON o.customer_id = c.customer_id
        JOIN items i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        COUNT(*) AS total_unique_customers,
        COUNT(*) FILTER (WHERE delivered_orders > 1) AS repeat_customers,
        COUNT(*) FILTER (WHERE delivered_orders = 1) AS one_time_customers,
        ROUND(
            (COUNT(*) FILTER (WHERE delivered_orders > 1) * 100.0) / NULLIF(COUNT(*), 0),
            2
        ) AS repeat_customer_rate_pct,
        ROUND(AVG(total_spent) FILTER (WHERE delivered_orders > 1), 2) AS avg_spend_repeat_customer,
        ROUND(AVG(total_spent) FILTER (WHERE delivered_orders = 1), 2) AS avg_spend_one_time_customer
    FROM customer_summary;
""").df()

q4_repeat_customers

,total_unique_customers,repeat_customers,one_time_customers,repeat_customer_rate_pct,avg_spend_repeat_customer,avg_spend_one_time_customer
0,93349,2801,90548,3.0,441.55,234.26


Business Insight:

Grouping by customer_unique_id rather than customer_id accurately identifies true individual repeat buyers across separate orders in the Olist schema.

Comparing repeat_customer_rate_pct alongside average spend per segment shows whether retention is a growth lever and how much cumulative value returning buyers generate compared to single-purchase shoppers.

# Query 5: Delivery Performance (On-Time vs. Late Deliveries)

In [16]:
"""How do actual delivery dates compare against estimated delivery dates for delivered orders, and what
percentage of orders fall into on-time versus late delivery buckets?"""

q5_delivery_performance = con.execute("""
    WITH delivery_variance AS (
        SELECT
            o.order_id,
            DATE_DIFF(
                'day',
                CAST(o.order_estimated_delivery_date AS TIMESTAMP),
                CAST(o.order_delivered_customer_date AS TIMESTAMP)
            ) AS delay_days
        FROM orders o
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
          AND o.order_estimated_delivery_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN delay_days <= 0 THEN 'On-Time or Early'
            WHEN delay_days BETWEEN 1 AND 3 THEN '1-3 Days Late'
            WHEN delay_days BETWEEN 4 AND 7 THEN '4-7 Days Late'
            ELSE '7+ Days Late'
        END AS delivery_bucket,
        COUNT(*) AS total_orders,
        ROUND((COUNT(*) * 100.0) / NULLIF(SUM(COUNT(*)) OVER (), 0), 2) AS order_share_pct,
        ROUND(AVG(delay_days), 1) AS avg_delay_days
    FROM delivery_variance
    GROUP BY 1
    ORDER BY total_orders DESC;
""").df()

q5_delivery_performance

,delivery_bucket,total_orders,order_share_pct,avg_delay_days
0,On-Time or Early,153104,93.21,-13.5
1,7+ Days Late,4905,2.99,19.7
2,1-3 Days Late,3209,1.95,1.8
3,4-7 Days Late,3034,1.85,5.5


Business Insight:

Measuring delay_days relative to order_estimated_delivery_date directly evaluates fulfillment reliability against customer expectations rather than raw transit time alone.

Identifying the share of orders in 4-7 Days Late and 7+ Days Late pinpoints where carrier bottlenecks or overly optimistic delivery promises risk hurting customer satisfaction and repeat purchases.

# Query 6: Payment Method & Installment Behavior

In [18]:
"""How is total transaction value distributed across payment methods (credit_card, boleto, voucher, debit_card),
and what is the average number of installments per payment type?"""

q6_payment_analysis = con.execute("""
    WITH payment_summary AS (
        SELECT
            p.payment_type,
            COUNT(DISTINCT p.order_id) AS total_orders,
            ROUND(AVG(p.payment_installments), 1) AS avg_installments,
            ROUND(SUM(p.payment_value), 2) AS total_payment_value
        FROM payments p
        JOIN orders o
            ON p.order_id = o.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        payment_type,
        total_orders,
        avg_installments,
        total_payment_value,
        ROUND(
            (total_payment_value * 100.0) / NULLIF(SUM(total_payment_value) OVER (), 0),
            2
        ) AS value_share_pct
    FROM payment_summary
    ORDER BY total_payment_value DESC;
""").df()

q6_payment_analysis

,payment_type,total_orders,avg_installments,total_payment_value,value_share_pct
0,credit_card,74298,3.5,20560351.96,78.49
1,boleto,19188,1.0,4699688.04,17.94
2,voucher,3679,1.0,590205.64,2.25
3,debit_card,1485,1.0,345040.41,1.32


Business Insight:

Combining value_share_pct with avg_installments shows how heavily buyers rely on installment payments to complete higher-value checkouts.

Understanding dominant payment methods and installment usage helps product and finance teams optimize checkout options, installment limits, and promotional offers.

# Query 7: Regional Revenue & Freight Cost Burden by State

In [19]:
"""Which customer states (customer_state) generate the highest
delivered product revenue and order volume, and
how does freight cost (freight_value) as a percentage of product price vary across states?"""

q7_regional_freight = con.execute("""
    WITH state_metrics AS (
        SELECT
            c.customer_state,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(i.price), 2) AS product_revenue,
            ROUND(SUM(i.freight_value), 2) AS total_freight_cost,
            ROUND(AVG(i.freight_value), 2) AS avg_freight_per_item
        FROM orders o
        JOIN customers c
            ON o.customer_id = c.customer_id
        JOIN items i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        customer_state,
        total_orders,
        product_revenue,
        total_freight_cost,
        avg_freight_per_item,
        ROUND(
            (total_freight_cost * 100.0) / NULLIF(product_revenue, 0),
            2
        ) AS freight_to_price_pct
    FROM state_metrics
    ORDER BY product_revenue DESC
    LIMIT 10;
""").df()

q7_regional_freight

,customer_state,total_orders,product_revenue,total_freight_cost,avg_freight_per_item,freight_to_price_pct
0,SP,40494,8600337.41,1193829.37,15.12,13.88
1,RJ,12350,2965531.65,501664.13,20.90,16.92
2,MG,11352,2651139.12,454818.30,20.60,17.16
3,RS,5345,1234825.59,226646.90,21.64,18.35
4,PR,4923,1124190.79,196918.89,20.41,17.52
5,SC,3546,860134.85,150082.67,21.54,17.45
6,BA,3256,841969.43,166624.77,26.54,19.79
7,DF,2080,507612.35,85186.98,20.95,16.78
8,GO,1957,474508.21,86620.72,22.32,18.25
9,ES,1995,459743.03,84089.19,22.16,18.29


Business Insight:

Comparing product_revenue against freight_to_price_pct shows whether high-revenue states also enjoy lower shipping cost ratios (typically closer to major fulfillment hubs) or face steep shipping overhead.

Spotting states with high freight_to_price_pct helps logistics and pricing teams evaluate regional inventory placement, shipping fee tiers, or free-shipping thresholds without eroding margins.

# Query 8: Order Status & Cancellation Rate by Product Category

In [20]:
"""Which product categories experience the highest share
 of cancelled or unavailable orders relative to total order attempts,
and how much potential revenue is lost to non-delivered orders?"""

q8_cancellation_by_category = con.execute("""
    WITH category_orders AS (
        SELECT
            COALESCE(t.product_category_name_english, p.product_category_name, 'Unknown') AS category,
            o.order_id,
            o.order_status,
            i.price
        FROM orders o
        JOIN items i
            ON o.order_id = i.order_id
        JOIN products p
            ON i.product_id = p.product_id
        LEFT JOIN cat_trans t
            ON p.product_category_name = t.product_category_name
    )
    SELECT
        category,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT order_id) FILTER (WHERE order_status = 'delivered') AS delivered_orders,
        COUNT(DISTINCT order_id) FILTER (WHERE order_status IN ('canceled', 'unavailable')) AS failed_orders,
        ROUND(
            (COUNT(DISTINCT order_id) FILTER (WHERE order_status IN ('canceled', 'unavailable')) * 100.0) /
            NULLIF(COUNT(DISTINCT order_id), 0),
            2
        ) AS failure_rate_pct,
        ROUND(
            SUM(price) FILTER (WHERE order_status IN ('canceled', 'unavailable')),
            2
        ) AS unfulfilled_order_value
    FROM category_orders
    GROUP BY 1
    HAVING COUNT(DISTINCT order_id) >= 50
    ORDER BY failure_rate_pct DESC
    LIMIT 10;
""").df()

q8_cancellation_by_category

,category,total_orders,delivered_orders,failed_orders,failure_rate_pct,unfulfilled_order_value
0,dvds_blu_ray,59,56,2,3.39,1439.90
1,fixed_telephony,217,212,3,1.38,3517.58
2,books_general_interest,512,496,7,1.37,2443.18
3,home_appliances_2,234,227,3,1.28,5336.00
4,musical_instruments,628,611,8,1.27,6249.31
5,small_appliances,630,609,8,1.27,4753.77
6,construction_tools_safety,167,159,2,1.20,955.50
7,costruction_tools_garden,194,190,2,1.03,197.52
8,Unknown,1451,1392,14,0.96,1663.07
9,fashion_male_clothing,112,106,1,0.89,98.00


Business Insight:

Filtering with HAVING COUNT(DISTINCT order_id) >= 50 avoids noisy percentages from tiny niche categories with very few orders.

High failure_rate_pct alongside meaningful unfulfilled_order_value points to stock-sync issues, inaccurate catalog listings, or seller fulfillment gaps that directly hurt conversion and trust.

# Query 9: Delivery Delay Impact on Customer Review Scores

In [21]:
"""How does delivery punctuality (On-Time / Early vs. Late) affect average customer review scores
(review_score) and the share of low ratings (1 or 2 stars)?"""

reviews = pd.read_csv("/content/olist_order_reviews_dataset.csv", engine="python", on_bad_lines="skip")
con.register("reviews", reviews)

q9_delivery_vs_reviews = con.execute("""
    WITH order_delivery_reviews AS (
        SELECT
            o.order_id,
            CASE
                WHEN DATE_DIFF(
                    'day',
                    CAST(o.order_estimated_delivery_date AS TIMESTAMP),
                    CAST(o.order_delivered_customer_date AS TIMESTAMP)
                ) <= 0 THEN 'On-Time or Early'
                ELSE 'Late Delivery'
            END AS delivery_status,
            r.review_score
        FROM orders o
        JOIN reviews r
            ON o.order_id = r.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
          AND o.order_estimated_delivery_date IS NOT NULL
          AND r.review_score IS NOT NULL
    )
    SELECT
        delivery_status,
        COUNT(*) AS total_reviewed_orders,
        ROUND(AVG(review_score), 2) AS avg_review_score,
        ROUND(
            (COUNT(*) FILTER (WHERE review_score <= 2) * 100.0) / NULLIF(COUNT(*), 0),
            2
        ) AS low_rating_share_pct,
        ROUND(
            (COUNT(*) FILTER (WHERE review_score = 5) * 100.0) / NULLIF(COUNT(*), 0),
            2
        ) AS five_star_share_pct
    FROM order_delivery_reviews
    GROUP BY 1
    ORDER BY avg_review_score DESC;
""").df()

q9_delivery_vs_reviews

,delivery_status,total_reviewed_orders,avg_review_score,low_rating_share_pct,five_star_share_pct
0,On-Time or Early,153114,4.29,9.32,62.24
1,Late Delivery,10930,2.27,62.43,16.76


Business Insight:

Comparing avg_review_score and low_rating_share_pct across On-Time or Early versus Late Delivery quantifies how logistics delays translate into customer dissatisfaction.

This connects operational SLA tracking directly to marketplace reputation—giving a strong data-backed argument for tighter delivery estimates or faster shipping lanes.

# Query 10: Top Sellers Performance & Delivery Speed (DENSE_RANK)

In [22]:
"""Who are the top 10 revenue-generating sellers on the marketplace, what share of total GMV do they drive,
and how fast do they ship orders compared to their delivery SLA?"""

q10_top_sellers = con.execute("""
    WITH seller_metrics AS (
        SELECT
            i.seller_id,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(i.price), 2) AS total_revenue,
            ROUND(AVG(
                DATE_DIFF(
                    'day',
                    CAST(o.order_purchase_timestamp AS TIMESTAMP),
                    CAST(o.order_delivered_customer_date AS TIMESTAMP)
                )
            ), 1) AS avg_delivery_days
        FROM orders o
        JOIN items i
            ON o.order_id = i.order_id
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
        GROUP BY 1
    )
    SELECT
        DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS seller_rank,
        seller_id,
        total_orders,
        total_revenue,
        ROUND((total_revenue * 100.0) / NULLIF(SUM(total_revenue) OVER (), 0), 2) AS revenue_share_pct,
        avg_delivery_days
    FROM seller_metrics
    ORDER BY seller_rank
    LIMIT 10;
""").df()

q10_top_sellers

,seller_rank,seller_id,total_orders,total_revenue,revenue_share_pct,avg_delivery_days
0,1,4869f7a5dfa277a7dca6462dcf3b52b2,1124,386596.70,1.72,15.1
1,2,53243585a1d6dc2643021fd1853d8905,348,373403.60,1.66,13.3
2,3,4a3ca9315b744ce9f8e9374361493884,1772,334049.11,1.49,14.4
3,4,fa1c13f2614d7b5c4749cbc52fecda94,578,318279.26,1.42,13.4
4,5,7c67e1448b00f6e969d365cea6b010ab,973,316003.84,1.41,22.2
5,6,7e93a43ef30c4f03f38b393420bc753a,319,283708.39,1.26,11.4
6,7,da8622b14eb17ae2831f4ac5b9dab84a,1311,270035.67,1.20,11.2
7,8,7a67c85e85bb2ce8582c35f2203ad736,1145,237004.10,1.06,11.3
8,9,1025f0e2d44d7041d6cf58b6550e0bfa,910,235225.32,1.05,12.0
9,10,955fee9216a65b617aa5c0531780ce60,1261,225252.10,1.00,10.8


Business Insight:

Ranking sellers by total_revenue alongside avg_delivery_days highlights whether top GMV contributors maintain fast fulfillment or create operational bottlenecks.

Identifying high-volume sellers with slower delivery times gives account managers a clear target for SLA coaching and logistics optimization.